# 03 Terminal 8k Matched-Packet Null

This notebook is the third evidence notebook in the public release suite.

## Purpose

It reproduces the **terminal 8k matched-packet placement null** and shows that:

- the real terminal 8k packet field remains tighter than a morphology-matched null ensemble
- the real field selects **`h1`** as the best harmonic
- the real field exceeds all **2048** matched-packet null replicates in the frozen release audit

This notebook is **artifact-first**. By default it loads frozen summary CSV/JSON artifacts and rebuilds the headline comparison table and simple release plots without rerunning the overnight null.


## Reading note

This notebook supports the claim that the terminal 8k concentration is **not explained by packet morphology alone**.

For the final robustness notebook, continue with:

- `04_terminal_8k_jackknife.ipynb`


In [ ]:
# Optional path settings for Colab or local runs

import os
import json
from pathlib import Path

DEFAULT_OUT_DIR = "/content/drive/MyDrive/Colab Notebooks/ForgeV16c"
OUT_DIR = os.environ.get("FORGE_V16C_OUT_DIR", DEFAULT_OUT_DIR)

print("OUT_DIR =", OUT_DIR)

## Expected frozen artifacts

This notebook looks for the following terminal 8k null artifacts:

- `s6_terminal8k_null_overnight_v1_summary.json`
- `s6_terminal8k_null_overnight_v1_harmonic_compare.csv`
- `s6_terminal8k_null_overnight_v1_replicates.csv`

If your filenames differ slightly, edit the `FILES` map below.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FILES = {
    "summary_json": os.path.join(OUT_DIR, "s6_terminal8k_null_overnight_v1_summary.json"),
    "harmonic_compare_csv": os.path.join(OUT_DIR, "s6_terminal8k_null_overnight_v1_harmonic_compare.csv"),
    "replicates_csv": os.path.join(OUT_DIR, "s6_terminal8k_null_overnight_v1_replicates.csv"),
}

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

for k, p in FILES.items():
    print(f"{k:>22}: {'FOUND' if os.path.exists(p) else 'MISSING'}")
    print(f"                        {p}")

## Load the frozen null artifacts

This cell loads the frozen terminal 8k null summary, the harmonic comparison table, and the replicate-level best-resultant table.


In [ ]:
summary = load_json(FILES["summary_json"])
harm_df = pd.read_csv(FILES["harmonic_compare_csv"])
rep_df = pd.read_csv(FILES["replicates_csv"])

display(pd.DataFrame([summary]))
display(harm_df)

## Headline interpretation

The matched-packet placement null preserves:

- packet count
- packet row lengths

but randomizes:

- packet locations among eligible terminal-board rows

The main release-level claim is that the real terminal 8k field remains **measurably tighter** than this morphology-matched null.


In [ ]:
headline = pd.DataFrame([{
    "real_best_candidate": summary.get("real_best_candidate"),
    "real_best_resultant_r": summary.get("real_best_resultant_r"),
    "real_best_arc80": summary.get("real_best_arc80"),
    "null_mean_best_r": summary.get("null_mean_best_r"),
    "null_mean_best_arc80": summary.get("null_mean_best_arc80"),
    "empirical_p_bestR_vs_matched_null": summary.get("empirical_p_bestR_vs_matched_null"),
    "n_nulls_ok": summary.get("n_nulls_ok"),
}])

display(headline)

## Harmonic-by-harmonic comparison

This table compares the real terminal 8k harmonic scan to the matched-null ensemble means and upper quantiles.


In [ ]:
display(harm_df[[
    "Candidate",
    "Harmonic",
    "Resultant_R",
    "Arc80",
    "Null_R_Mean",
    "Null_R_Q95",
    "Null_Arc80_Mean",
    "Null_Arc80_Q95",
]])

## Plot the matched-null comparison

Panel A compares real vs null harmonic resultants.  
Panel B compares the real best resultant to the null replicate distribution.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

ax = axes[0]
x = np.arange(len(harm_df))
ax.bar(x - 0.2, harm_df["Resultant_R"], width=0.4, label="Real 8k")
ax.bar(x + 0.2, harm_df["Null_R_Mean"], width=0.4, label="Matched-null mean")
ax.plot(x + 0.2, harm_df["Null_R_Q95"], marker="o", linestyle="None", label="Matched-null 95th pct")
ax.set_xticks(x)
ax.set_xticklabels(harm_df["Candidate"], rotation=25, ha="right")
ax.set_ylabel("Resultant R")
ax.set_title("A. Real vs matched-packet null by harmonic")
ax.legend()

ax = axes[1]
good_rep_df = rep_df.copy()
if "placement_failed" in good_rep_df.columns:
    good_rep_df = good_rep_df[~good_rep_df["placement_failed"].fillna(False)]

ax.hist(good_rep_df["best_resultant_r"], bins=30)
ax.axvline(summary["real_best_resultant_r"], linestyle="--", linewidth=2, label="Real best R")
ax.set_xlabel("Best resultant R across harmonics")
ax.set_ylabel("Count")
ax.set_title("B. Matched-packet null best-R distribution")
ax.legend()

plt.tight_layout()
plt.show()

## Simple release checks

These are the release-level checks this notebook should satisfy:

- the real best harmonic is `logT_over_pi_h1`
- the real best resultant exceeds the null mean
- the real best Arc80 is smaller than the null mean Arc80
- the empirical null exceedance probability is approximately `1 / 2049`


In [ ]:
checks = {
    "real_best_is_h1": summary.get("real_best_candidate") == "logT_over_pi_h1",
    "real_R_gt_null_mean": float(summary.get("real_best_resultant_r")) > float(summary.get("null_mean_best_r")),
    "real_Arc80_lt_null_mean": float(summary.get("real_best_arc80")) < float(summary.get("null_mean_best_arc80")),
    "p_floor_matches_2048_run": abs(float(summary.get("empirical_p_bestR_vs_matched_null")) - (1.0 / 2049.0)) < 1e-12,
}

checks_df = pd.DataFrame([checks])
display(checks_df)

if checks_df.all(axis=None):
    print("All terminal 8k matched-packet null release checks passed.")
else:
    print("One or more terminal 8k null release checks failed. Inspect the frozen artifacts.")

## Optional export

If you want to save a compact release summary into the repository artifacts folder, run the next cell.


In [ ]:
# Optional export
# release_null_summary = headline.copy()
# export_path = os.path.join(OUT_DIR, "release_terminal8k_matched_packet_null_summary.csv")
# release_null_summary.to_csv(export_path, index=False)
# print("Saved:", export_path)

## Heavy recomputation note

The overnight matched-packet null is intentionally **not rerun by default** in this public notebook.  
This notebook is meant to read and explain the frozen release artifacts.

If you later decide to expose the heavy rerun path, add it as a clearly marked optional section at the end of the notebook.


## Next notebook

Continue to:

**`04_terminal_8k_jackknife.ipynb`**

That notebook reproduces the 10,000-run jackknife robustness audit and shows that the terminal 8k lock is not driven by a tiny outlier subset.
